In [ ]:
import arcgis
import time
from arcgis.gis import GIS
from arcgis.gis import Item
from arcgis.apps.storymap import StoryMap

from typing import Set  # Import Set from typing
import re, json, csv

import pandas as pd
import os
import logging
import requests

# Set Pandas dataframe display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns',1000)

In [ ]:
agoNotebook = False
# Print the version of the arcgis module
print(f"Running ArcGIS API for Python version: {arcgis.__version__}")

# Define the GIS
if agoNotebook == False:
    import keyring
    service_name = "system" # Use the default local credential store
    success = False # Set initial state

    # Ask for the username
    while success == False:
        username_for_keyring = input("Enter your ArcGIS Online username:") # If you are using VS Code, the text input dialog box appears at the top of the window
        # Get the credential object
        credential = keyring.get_credential(service_name, username_for_keyring)
        # Check if the username is in the credential store
        if credential is None:
            print(f"'{username_for_keyring}' is not in the local system's credential store. Try another username.")
        # Retrieve the password, login and set the GIS portal
        else:
            password_from_keyring = keyring.get_password("system", username_for_keyring)
            portal_url = 'https://www.arcgis.com'  
            gis = GIS(portal_url, username=username_for_keyring, password=password_from_keyring)
            success = True
            # Print a success message with username and user's organization role
            print("Successfully logged in as: " + gis.properties.user.username, "(role: " + gis.properties.user.role + ")")
else:
    gis = GIS("home")

In [ ]:
classic_maptour_id = "73f4483f851b4f8f92eb4efedaf98957"
# classic_maptour_webmap = ""
# classic_maptour_featureCollection = ""
# classic_maptour_featureSet = ""

In [ ]:
from converter_json import *

# Retrieve the JSON data for the classic item
classic_item = gis.content.get(classic_maptour_id)
classic_json = classic_item.get_data()


In [ ]:
target_story_id, new_storymap_json = convert_classic_to_json(classic_json, theme_id="summit", gis_token=gis._con.token, gis=gis)

In [ ]:
print(target_story_id)

In [ ]:
with open("map-tour-converted.json", "w", encoding="utf-8") as f:
    json.dump(new_storymap_json, f, indent=4, ensure_ascii=False)

In [ ]:
# Assume you have:
username = gis.properties.user.username
token = gis._con.token

# 1. Find the draft resource name (e.g., "draft_*.json")
resources_url = f"https://www.arcgis.com/sharing/rest/content/items/{target_story_id}/resources?f=json&token={token}"
resources_response = requests.get(resources_url).json()
for res in resources_response.get("resources", []):
    resource_name = res.get("resource", "")
    if resource_name.startswith("draft_") and resource_name.endswith(".json"):
        draft_resource_name = resource_name
        break
if not draft_resource_name:
    raise Exception("Could not find draft resource in target StoryMap.")

# 2. Remove the old draft resource
remove_resource_url = f"https://www.arcgis.com/sharing/rest/content/users/{username}/items/{target_story_id}/removeResources"
remove_params = {
    "f": "json",
    "token": token,
    "resource": draft_resource_name
}
remove_response = requests.post(remove_resource_url, data=remove_params)
assert remove_response.json().get("success"), "Failed to remove old draft resource"

# 3. Upload the new draft resource
add_resource_url = f"https://www.arcgis.com/sharing/rest/content/users/{username}/items/{target_story_id}/addResources"
json_blob = json.dumps(new_storymap_json)
files = {"file": (draft_resource_name, json_blob)}
add_params = {
    "f": "json",
    "token": token,
    "fileName": draft_resource_name
}
add_response = requests.post(add_resource_url, files=files, data=add_params)
assert add_response.json().get("success"), "Failed to upload new draft resource"

print(f"Updated {draft_resource_name} in StoryMap {target_story_id}")